# Day 2 · 8교시 [실습 보조] PostgreSQL 직접 다루기 — `08_postgres_db`

## 실습 목표

교안 8교시에서 에이전트는 **MCP**로 DB를 다룬다 — 스키마 생성, 쿼리, 마이그레이션까지.
이 노트북은 그 **아래층에서 벌어지는 일**을 파이썬+SQL로 직접 해 본다. MCP 도구가
블랙박스가 되지 않도록: *에이전트가 대신해 주는 일의 정체*를 한 번 손으로 겪는 것이 목적이다.

| 순서 | 내용 | 교안 연결 |
|------|------|----------|
| 1 | ERD → 실제 스키마 (CREATE TABLE) | 4교시 ERD · 8.6 |
| 2 | INSERT + **중복 방지**(url UNIQUE) | 유스케이스 예외 흐름 · 5교시 테스트 명세 |
| 3 | 조회·집계 (GROUP BY) | 8.7 자연어 쿼리의 실체 |
| 4 | 마이그레이션 (ALTER TABLE) | 8.7 · 문서 동기화 |
| 5 | **자연어→SQL 미니 구현** (LLM) | MCP 도구가 하는 일의 정체 |

> ⚙️ 준비: PostgreSQL 컨테이너 (교안 부록 B)
> ```bash
> docker run -d --name vibe-pg -e POSTGRES_PASSWORD=pass -e POSTGRES_DB=vibe -p 5434:5432 postgres:16-alpine
> ```
> 의존성: `pip install psycopg2-binary pandas python-dotenv openai`
> LLM(5절)은 루트 `.env`의 `MLAPI_*` 키를 쓴다 — 없으면 5절만 건너뛴다(graceful).

In [1]:
import warnings, psycopg2
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")   # 실습 출력 청결용

# 접속 정보 — 자기 환경에 맞게 (위 docker run 기준 5434, 로컬 설치면 보통 5432)
PG = dict(host="localhost", port=5434, dbname="vibe", user="postgres", password="pass")

conn = psycopg2.connect(**PG)
conn.autocommit = True
cur = conn.cursor()
cur.execute("SELECT version()")
print("접속 OK:", cur.fetchone()[0].split(" on ")[0])

접속 OK: PostgreSQL 16.14


## 1. ERD → 실제 스키마

4교시 ERD(`docs/ERD.md`)의 두 테이블을 그대로 DDL로 옮긴다. 핵심은 **`url UNIQUE`** —
"같은 글을 두 번 수집하지 않는다"는 유스케이스 예외 흐름이 **제약 조건**이라는 실물이 된다.
(재실행 가능하도록 DROP 후 생성)

In [2]:
cur.execute("DROP TABLE IF EXISTS summaries, items CASCADE")
cur.execute("""
CREATE TABLE items (
    id         SERIAL PRIMARY KEY,
    source     TEXT NOT NULL,
    url        TEXT NOT NULL UNIQUE,          -- 중복 수집 방지 (ERD의 UK)
    title      TEXT NOT NULL,
    content    TEXT,
    created_at TIMESTAMP DEFAULT now()
)""")
cur.execute("""
CREATE TABLE summaries (
    id          SERIAL PRIMARY KEY,
    target_date DATE NOT NULL,
    body        TEXT NOT NULL,
    sent_at     TIMESTAMP                      -- 발송했는가(NULL=미발송)
)""")
cur.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='public'")
print("생성된 테이블:", [r[0] for r in cur.fetchall()])

생성된 테이블: ['items', 'summaries']


## 2. INSERT + 중복 방지 — 제약이 명세다

수집 데이터 3건을 넣고, **같은 url 을 다시** 넣어 본다. `ON CONFLICT (url) DO NOTHING` 으로
중복은 조용히 스킵된다 — 5교시에서 `save_items` 테스트가 요구한 "중복 url 제외, 신규 건수 반환"이
바로 이 동작의 명세였다.

In [3]:
SAMPLE = [
    ("hn",   "https://example.com/a", "에이전트 시대의 개발", "본문 A"),
    ("hn",   "https://example.com/b", "MCP 표준의 확산",     "본문 B"),
    ("blog", "https://example.com/c", "vibe coding 후기",    "본문 C"),
]

def save_items(rows):
    """신규 저장 건수를 반환 (중복 url 은 스킵) — 5교시 테스트의 그 함수."""
    n = 0
    for source, url, title, content in rows:
        cur.execute("""INSERT INTO items (source, url, title, content)
                       VALUES (%s, %s, %s, %s) ON CONFLICT (url) DO NOTHING""",
                    (source, url, title, content))
        n += cur.rowcount            # 스킵되면 0
    return n

print("1차 저장:", save_items(SAMPLE), "건 (신규 3)")
print("2차 저장:", save_items(SAMPLE[:1]), "건 (같은 url → 신규 0)")
cur.execute("SELECT count(*) FROM items"); print("총 행 수:", cur.fetchone()[0], "(중복 없이 3)")

1차 저장: 3 건 (신규 3)
2차 저장: 0 건 (같은 url → 신규 0)
총 행 수: 3 (중복 없이 3)


## 3. 조회·집계 — "소스별로 몇 건?"

교안 8.7에서 에이전트에게 자연어로 시킨 질문의 **SQL 실체**가 이것이다.

In [4]:
import pandas as pd

df = pd.read_sql("SELECT source, count(*) AS cnt FROM items GROUP BY source ORDER BY cnt DESC", conn)
print(df.to_string(index=False))

source  cnt
    hn    2
  blog    1


## 4. 마이그레이션 — 스키마는 변한다

"요약 완료 여부를 추적하고 싶다" → 칼럼 추가. 실물(DB)을 바꾸면 **문서(ERD)도 함께** 갱신해야
설계문서가 살아있는 문서로 남는다(교안 8.7).

In [5]:
cur.execute("ALTER TABLE items ADD COLUMN IF NOT EXISTS summarized BOOLEAN DEFAULT FALSE")
cur.execute("""SELECT column_name, data_type FROM information_schema.columns
               WHERE table_name='items' ORDER BY ordinal_position""")
for name, typ in cur.fetchall():
    print(f"  items.{name:12} {typ}")
print("→ docs/ERD.md 에도 summarized 칼럼을 반영해야 문서와 실물이 일치한다")

  items.id           integer
  items.source       text
  items.url          text
  items.title        text
  items.content      text
  items.created_at   timestamp without time zone
  items.summarized   boolean
→ docs/ERD.md 에도 summarized 칼럼을 반영해야 문서와 실물이 일치한다


## 5. 자연어→SQL 미니 구현 — MCP 도구의 정체

교안 8.7에서 "소스별로 집계해줘"라고 하면 에이전트가 SQL을 만들어 실행했다. 그 흐름을
직접 구현한다: **① 스키마를 읽어 → ② LLM에 질문과 함께 주고 → ③ 나온 SQL을 실행**.
MCP PostgreSQL 서버가 하는 일이 본질적으로 이것이다(+표준 프로토콜·권한 관리).

> 🛡️ 안전장치: 생성된 SQL이 **SELECT 로 시작할 때만** 실행한다 — 교안 8.8(권한은 좁게)의
> 미니 버전. LLM이 만든 문자열을 무조건 실행하는 것은 인젝션 문(Day 3 보안)을 여는 일이다.

In [6]:
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path().resolve().parents[1] / ".env")   # 루트 .env (MLAPI_*)

def get_schema():
    cur.execute("""SELECT table_name, column_name, data_type FROM information_schema.columns
                   WHERE table_schema='public' ORDER BY table_name, ordinal_position""")
    lines = {}
    for t, c, d in cur.fetchall():
        lines.setdefault(t, []).append(f"{c} {d}")
    return "\n".join(f"{t}({', '.join(cols)})" for t, cols in lines.items())

def nl_to_sql(question):
    """자연어 질문 → SQL 생성(LLM) → SELECT 만 실행 → 결과 반환."""
    from openai import OpenAI
    client = OpenAI(base_url=os.getenv("MLAPI_BASE_URL"), api_key=os.getenv("MLAPI_API_KEY"))
    prompt = (f"PostgreSQL 스키마:\n{get_schema()}\n\n질문: {question}\n"
              "위 질문에 답하는 SQL 한 문장만 출력해. 설명·코드블록 없이 SQL만.")
    r = client.chat.completions.create(
        model=os.getenv("MLAPI_MODEL", "openai/gpt-5-mini"),
        max_completion_tokens=2000,
        messages=[{"role": "user", "content": prompt}])
    sql = r.choices[0].message.content.strip().strip("`").removeprefix("sql").strip()
    print("생성된 SQL:", sql)
    assert sql.lower().startswith("select"), "SELECT 외에는 실행하지 않는다(안전장치)"
    return pd.read_sql(sql, conn)

if os.getenv("MLAPI_API_KEY"):
    print(nl_to_sql("소스별 수집 건수를 많은 순으로 보여줘").to_string(index=False))
else:
    print("MLAPI_* 키 없음 → 5절 건너뜀 (graceful)")

생성된 SQL: SELECT source, COUNT(*) AS cnt FROM items GROUP BY source ORDER BY cnt DESC;
source  cnt
    hn    2
  blog    1


## 실습 정리

- **ERD→스키마→제약**: 설계문서의 문장("중복 수집 안 함")이 `UNIQUE` 제약이라는 실물이 됐다.
- **중복 방지·집계·마이그레이션**을 SQL로 직접 — 에이전트가 MCP로 해 주는 일의 밑바닥.
- **자연어→SQL**: 스키마+질문→LLM→SQL→(SELECT만) 실행 — MCP DB 도구의 본질 + 최소 안전장치.
- 이제 교안 8교시로 돌아가 **같은 일을 에이전트에게** 시켜 보라(MCP) — 아래층을 알고 쓰는
  도구는 블랙박스가 아니다. 스키마 변경 시 ERD 문서 동기화를 잊지 말 것.